# Checkpoint 5: Cross-Validation

Checking how stable each model's performance really is using k-fold cross-validation.


## 1. Restore the works

In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 60)


In [55]:
df = pd.read_csv("cars.csv")

In [56]:
df

,id_x,car_rel_url_x,datetime_scrape,name,price_x,currency_x,datetime_product,city,day,hour,attributes,production_year,engine_displacement_num,engine_displacement_unit,kilometrage_num,kilometrage_unit,barter,loan,salon,spare_parts,vip,featured,img_url,id_y,cars_id,car_rel_url_y,datetime,description,price_y,currency_y,owner_name,shop_name,phone,updated,views,vin,car_details_id_x,Ban növü,Buraxılış ili,Hansı bazar üçün yığılıb,Marka,Model,Mühərrik,Qəzalı,Rəng,Sahiblər,Sürətlər qutusu,Vəziyyəti,Yeni,Yerlərin sayı,Yürüş,Ötürücü,Şəhər,car_details_id_y,car_rel_url,extra_info
0,3c234145-d57a-4ad6-9448-d43810fc3392,/autos/8748840-hyundai-i30,2024-09-13 20:32:19.751157+00,Hyundai i30,15000.0,AZN,"Bakı, dünən 23:28",bakı,13.09.2024,23:28,"2008, 1.6 L, 270 000 km",2008,1.6,L,270000,km,NaN,NaN,NaN,NaN,NaN,NaN,https://turbo.azstatic.com/uploads/f460x343/20...,8d84d800-fafd-4d5c-b640-f47bd6c5ac20,3c234145-d57a-4ad6-9448-d43810fc3392,/autos/8748840-hyundai-i30,2024-09-13 20:40:28.618345+00,Salam orjinal probeqdir bir ildi bizdedir biri...,15000.0,AZN,Şəmi,NaN,507687355.0,13.09.2024,492,NaN,8d84d800-fafd-4d5c-b640-f47bd6c5ac20,"Hetçbek, 5 qapı",2008,NaN,Hyundai,i30,1.6 L/115 a.g./Dizel,NaN,Gümüşü,2,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Xeyr,5,270 000 km,Ön,Bakı,8d84d800-fafd-4d5c-b640-f47bd6c5ac20,/autos/8748840-hyundai-i30,Yüngül lehimli disklər* ABS* Mərkəzi qapanma* ...
1,c74ea36f-6be1-4de4-926d-e117197dcf00,/autos/8475807-lada-vaz-niva-travel,2024-09-13 20:32:19.751157+00,LADA (VAZ) Niva Travel,23700.0,AZN,"Bakı, dünən 23:59",bakı,13.09.2024,23:59,"2024, 1.7 L, 0 km",2024,1.7,L,0,km,NaN,NaN,Salon,NaN,vipped-icon,featured-icon,https://turbo.azstatic.com/uploads/f460x343/20...,2cf8b84b-adaf-467a-8f06-3dabcf866c8a,c74ea36f-6be1-4de4-926d-e117197dcf00,/autos/8475807-lada-vaz-niva-travel,2024-09-13 20:40:28.618345+00,LADA Niva Travel modelini nəğd və ya sərfəli l...,23700.0,AZN,NaN,Lada Azərbaycan,554092445.0,13.09.2024,60189,NaN,2cf8b84b-adaf-467a-8f06-3dabcf866c8a,"Offroader / SUV, 5 qapı",2024,Rəsmi diler,LADA (VAZ),Niva Travel,1.7 L/80 a.g./Benzin,NaN,Yaşıl,NaN,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Bəli,5,0 km,Tam,Bakı,2cf8b84b-adaf-467a-8f06-3dabcf866c8a,/autos/8475807-lada-vaz-niva-travel,Yüngül lehimli disklər* ABS* Kondisioner* Otur...
2,9cefceb0-024d-4581-a869-a3c2c68a9f95,/autos/8739686-toyota-land-cruiser,2024-09-13 20:32:19.751157+00,Toyota Land Cruiser,35600.0,$,"Bakı, dünən 23:59",bakı,13.09.2024,23:59,"2011, 4.0 L, 164 750 km",2011,4.0,L,164750,km,NaN,NaN,NaN,NaN,vipped-icon,featured-icon,https://turbo.azstatic.com/uploads/f460x343/20...,22bb3704-ebe7-4357-ba2f-1806d1a7042b,9cefceb0-024d-4581-a869-a3c2c68a9f95,/autos/8739686-toyota-land-cruiser,2024-09-13 20:40:28.618345+00,Bezkraska.Mashinda hec bir problem yoxdur.Alve...,35600.0,USD,Anar,NaN,502126242.0,13.09.2024,2473,NaN,22bb3704-ebe7-4357-ba2f-1806d1a7042b,"Offroader / SUV, 5 qapı",2011,Rəsmi diler,Toyota,Land Cruiser,4.0 L/282 a.g./Benzin,NaN,Ağ,0,Avtomat,"Vuruğu yoxdur, rənglənməyib",Xeyr,8+,164 750 km,Tam,Bakı,22bb3704-ebe7-4357-ba2f-1806d1a7042b,/autos/8739686-toyota-land-cruiser,Yüngül lehimli disklər* ABS* Lyuk* Mərkəzi qap...
3,459cc337-fb63-48de-9694-41554923d311,/autos/8712597-hyundai-elantra,2024-09-13 20:32:19.751157+00,Hyundai Elantra,26700.0,AZN,"Bakı, dünən 23:59",bakı,13.09.2024,23:59,"2018, 2.0 L, 126 000 km",2018,2.0,L,126000,km,NaN,NaN,NaN,NaN,vipped-icon,NaN,https://turbo.azstatic.com/uploads/f460x343/20...,e0d16dac-4091-4417-916e-cadeff600f95,459cc337-fb63-48de-9694-41554923d311,/autos/8712597-hyundai-elantra,2024-09-13 20:40:28.618345+00,2019 alış\n,26700.0,AZN,Babək,NaN,774999004.0,13.09.2024,3727,5NPD84LFXKH406133,e0d16dac-4091-4417-916e-cadeff600f95,Sedan,2018,NaN,Hyundai,Elantra,2.0 L/150 a.g./Benzin,NaN,Boz,1,Avtomat,"Vuruğu yoxdur, rənglənməyib",Xeyr,NaN,126 000 km,Ön,Bakı,e0d16dac-4091-4417-916e-cadeff600f95,/autos/8712597-hyundai-elantra,Yüngül lehimli disklər* ABS* Lyuk* Yağış senso...
4,6c5ee8d8-1c6f-4fad-a694-957a4c43c25d,/autos/8674773-toyo

In [57]:
# Remove duplicate listings
df = df.drop_duplicates(subset = "id_x").copy()

In [58]:
# Build a single price column in AZN using fixed exchange rates
FX = {"AZN": 1.0, "$": 1.70, "€": 1.85}
df["price_azn"] = df["price_x"] * df["currency_x"].map(FX)
df = df[df["price_azn"].notnull()].copy()

In [59]:
feature_columns = [
    "production_year",
    "engine_displacement_num",
    "kilometrage_num",
    "views",
    "city",
    "Ban növü",
    "Marka",
    "Rəng",
    "Sahiblər",
    "Sürətlər qutusu",
    "Vəziyyəti",
    "Yerlərin sayı",
    "Ötürücü",
]
target_column = "price_azn"

In [60]:
df_model = df[feature_columns + [target_column]].copy()


In [61]:
X = df_model[feature_columns]
y = df_model[target_column]

In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [63]:
numeric_features = ["production_year", "engine_displacement_num", "kilometrage_num", "views"]
categorical_features = ["city", "Ban növü", "Marka", "Rəng", "Sahiblər",
                         "Sürətlər qutusu", "Vəziyyəti", "Yerlərin sayı", "Ötürücü"]

In [64]:
numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

In [65]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
])

models = {
    "Linear Regression": linear_model,
    "Random Forest": random_forest_model,
}

In [66]:
print("X_train:", X_train.shape)

X_train: (519439, 13)


## 2. A smaller sample for cross-validation

Cross-validation means fitting each model several times instead of once, which takes several times as long.

Running 5-fold cross-validation on the entire training set would be computationally expensive due to the dataset size.

To keep execution time reasonable, cross-validation is performed on a random sample of 80,000 training observations.

In [67]:
cv_sample_size = 80_000
rng = np.random.RandomState(42)
sample_idx = rng.choice(X_train.index,
                        size = cv_sample_size,
                        replace = False)

In [68]:
X_cv = X_train.loc[sample_idx]
y_cv = y_train.loc[sample_idx]

print("Cross-validation sample:", X_cv.shape)

Cross-validation sample: (80000, 13)


## 3. Running 5-fold cross-validation for both models



In [69]:
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

In [71]:
cv_results = {}

for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_cv, y_cv, cv = kf, scoring = scoring, n_jobs = 1)
    cv_results[name] = scores
    print(f"{name}: done.")


Linear Regression: done.
Random Forest: done.


## 4. Summarize the cross-validation results

In [82]:
summary_rows = []

for name, scores in cv_results.items():
    mae_scores = -scores["test_MAE"]
    rmse_scores = -scores["test_RMSE"]
    r2_scores = scores["test_R2"]
    summary_rows.append({
        "model": name,
        "MAE_mean": round(mae_scores.mean(), 0),
        "MAE_std": round(mae_scores.std(), 0),
        "RMSE_mean": round(rmse_scores.mean(), 0),
        "RMSE_std": round(rmse_scores.std(), 0),
        "R2_mean": round(r2_scores.mean(), 3),
        "R2_std": round(r2_scores.std(), 3),
    })


In [83]:
cv_summary_df = pd.DataFrame(summary_rows)
cv_summary_df

,model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,Linear Regression,17787.0,176.0,34862.0,1111.0,0.525,0.026
1,Random Forest,6188.0,47.0,12681.0,192.0,0.937,0.002


## 5. Written interpretation

Cross-validation provides a more reliable estimate of model performance than a single train/test split because the model is evaluated on multiple data partitions.

Consistent scores across folds (low standard deviation) indicate that the model generalizes well and is less sensitive to how the data is split.

Comparing these results with the hold-out test evaluation helps verify whether the selected model performs consistently on unseen data.


In [84]:
best_model = cv_summary_df.sort_values("RMSE_mean").iloc[0]

print(f"Best model based on cross-validation RMSE: {best_model['model']}")

Best model based on cross-validation RMSE: Random Forest
